In [ ]:
# Modo de ejecución: "validacion" entrena solo con train (permite medir nDCG),
# "entrega" entrena con train+test (usa todo el historial disponible para predecir).
#MODO = "validacion"
MODO = "entrega"

#### 1. Librerías.

In [4]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [5]:
%run "./constantes/constantes.ipynb"

In [ ]:
# Verificación del modo (que los paths coincidan con lo que creo que estoy corriendo).
print(f"MODO: {MODO} | sufijo: {sufijo!r}")
print(f"train_fe: {path_train_fe}")
print(f"modelo:   {path_modelo}")

#### 3. Funciones.

In [6]:
%run "./funciones/funciones.ipynb"

#### 4. Lecturas.

In [7]:
#a. Train.
df_train = pd.read_csv(path_train_fe)

In [8]:
#b. Test.
df_test = pd.read_csv(path_test_crudo)

In [9]:
#c. Dataset a predecir.
df_a_predecir = pd.read_csv(path_a_predecir)

In [10]:
#d. Libros y Lectores (lo tomo para armar la predicción).
df_libros = pd.read_csv(path_libros_fe)
df_lectores = pd.read_csv(path_lectores_fe)

In [11]:
#e. Leo el modelo.
rf = joblib.load(path_modelo)

#### 5. Preparación previa.

In [12]:
#c. Armo X e y. 
features_base = [
    "anio_edicion", 
    "nacimiento",
    "edad_al_interactuar", 
    #"dias_transcurridos_interaccion",
    "anios_transcurridos_edicion", 
    #"antiguedad_libro_hoy",
    'frecuencia_lector', 
    'frecuencia_libro', 
    'n_interacciones_lector_autor',
    'n_interacciones_lector_genero', 
    'n_lectores_distintos_autor',
    #'rating_prom_id_lector', 
    'rating_prom_id_lector_autor',
    'rating_prom_id_lector_genero_libro_agrupado', 
    #'rating_prom_id_libro',
    #'rating_prom_autor', 
    #'rating_prom_genero', 
    'n_autores_distintos_lector',
    'n_generos_distintos_lector'
]
features_dummies = [c for c in df_train.columns if c.startswith((
    "genero_persona_", 
    #"genero_libro_agrupado_", 
    #"editorial_agrupada_", 
    #"pais_persona_agrupado"
))]
features = features_base + features_dummies

#### 6. Predicción.

In [13]:
#a. Precómputos (fuera del loop).
#1. Universo de libros.
conn = sqlite3.connect(path_db)
todos_los_libros = pd.read_sql("SELECT id_libro FROM interacciones", conn)["id_libro"].unique()
conn.close()
#2. Historial por lector.
leidos_por_lector = df_train[["id_lector","id_libro"]].groupby("id_lector")["id_libro"].apply(set).to_dict()
#3. Lectores a recomendarle.
id_lectores_predecir = df_a_predecir["id_lector"].unique()

In [14]:
#4. Verificación: los lectores a predecir, ¿tienen historial?
freq = df_train.groupby("id_lector").size()
cobertura = df_a_predecir["id_lector"].map(freq)
print(f"Modo: {MODO} | Filas de la base: {len(df_train)}")
print("Lectores pedidos:", len(id_lectores_predecir))
print("Sin historial:", cobertura.isna().sum())
print("Mediana de frecuencia:", cobertura.median())

Modo: entrega | Filas de la base: 461407
Lectores pedidos: 832
Sin historial: 320
Mediana de frecuencia: 95.0


In [15]:
#b. Variables que me van a servir para el feature engineering de test.
#i.Media global del rating en TRAIN.
media_global = df_train["rating"].mean()

#ii. Características del lector.
x_caract_lector_base = [
    "id_lector",
    #"nombre",
    #"vive_en",
    "nacimiento",
    "ciudad",
    "pais"
] 

caract_lector_base = df_lectores[x_caract_lector_base].drop_duplicates("id_lector")

# Las dummies las armo desde df_lectores (tiene TODOS los lectores), no desde df_train
# (solo tiene los que interactuaron). Si no, todo lector sin historial queda en NaN.
caract_lector_dummies = pd.get_dummies(
    df_lectores[["id_lector", "genero_persona"]].drop_duplicates("id_lector"),
    columns=["genero_persona"]
)

# Alineo con las columnas exactas que vio el modelo en train:
# si aparece una categoría que train no tenía, la descarto; si falta una que espera, la creo en 0.
cols_dummies_train = [c for c in df_train.columns if c.startswith("genero_persona_")]
caract_lector_dummies = caract_lector_dummies.reindex(
    columns=["id_lector"] + cols_dummies_train,
    fill_value=0
)

caract_lector = caract_lector_base.merge(caract_lector_dummies,how="left",on="id_lector")

# Red de seguridad: si algún lector no matcheó en el merge, las dummies quedan en 0.
caract_lector[cols_dummies_train] = caract_lector[cols_dummies_train].fillna(0)

# Frecuencia del lector para TEST ----> Se calcula sobre todo df_train, sin LOO.
freq_lector_test = (
    df_train
    .groupby("id_lector")
    .size()
)

caract_lector["frecuencia_lector"] = (
    caract_lector["id_lector"]
    .map(freq_lector_test)
    .fillna(0)
)

# Rating promedio del lector para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_lector_test = (
    df_train
    .groupby("id_lector")["rating"]
    .mean()
)

caract_lector["rating_prom_id_lector"] = (
    caract_lector["id_lector"]
    .map(means_lector_test)
    .fillna(media_global)
)

# Cantidad de autores distintos que leyó para TEST ----> Se calcula sobre todo df_train, sin LOO.
autores_por_lector_test = (
    df_train
    .groupby("id_lector")["autor"]
    .nunique()
)

caract_lector["n_autores_distintos_lector"] = (
    caract_lector["id_lector"]
    .map(autores_por_lector_test)
    .fillna(0)
)

# Cantidad de géneros distintos que leyó para TEST ----> Se calcula sobre todo df_train, sin LOO.
generos_por_lector_test = (
    df_train
    .groupby("id_lector")["genero_libro_agrupado"]
    .nunique()
)

caract_lector["n_generos_distintos_lector"] = (
    caract_lector["id_lector"]
    .map(generos_por_lector_test)
    .fillna(0)
)

#iii. Comprobación: no puede quedar ningún nulo.
nulos = caract_lector.drop(columns=["ciudad", "pais"]).isna().sum()
print("Nulos en caract_lector:")
print(nulos[nulos > 0] if nulos.sum() else "Ninguno.")
print("Lectores:", len(caract_lector), "| Duplicados:", caract_lector["id_lector"].duplicated().sum())

Nulos en caract_lector:
Ninguno.
Lectores: 11285 | Duplicados: 0


In [16]:
#iii. Características de los libros.
x_caract_libros_base = [
    "id_libro",
    "autor",
    "genero_libro_agrupado",
    #"editorial",
    "anio_edicion",
    #"isbn",
    #"resumen"
]
caract_libros_base = df_libros[x_caract_libros_base].drop_duplicates("id_libro")

# Las dummies las armo desde df_libros (tiene TODO el catálogo), no desde df_train
# (solo tiene los libros que alguien leyó). Si no, todo libro sin interacciones queda en NaN.
caract_libros_dummies = pd.get_dummies(
    df_libros[["id_libro", "genero_libro_agrupado"]].drop_duplicates("id_libro"),
    columns=["genero_libro_agrupado"]
)

# Alineo con las columnas exactas que vio el modelo en train:
# si aparece una categoría que train no tenía, la descarto; si falta una que espera, la creo en 0.
cols_dummies_libro_train = [c for c in df_train.columns if c.startswith("genero_libro_agrupado_")]
caract_libros_dummies = caract_libros_dummies.reindex(
    columns=["id_libro"] + cols_dummies_libro_train,
    fill_value=0
)

caract_libros = caract_libros_base.merge(caract_libros_dummies,how="left",on="id_libro")

# Red de seguridad: si algún libro no matcheó en el merge, las dummies quedan en 0.
caract_libros[cols_dummies_libro_train] = caract_libros[cols_dummies_libro_train].fillna(0)

# Frecuencia del libro para TEST ----> Se calcula sobre todo df_train, sin LOO.
freq_libro_test = (
    df_train
    .groupby("id_libro")
    .size()
)

caract_libros["frecuencia_libro"] = (
    caract_libros["id_libro"]
    .map(freq_libro_test)
    .fillna(0)
)

# Cantidad de lectores distintos del autor para TEST ----> Se calcula sobre todo df_train, sin LOO.
pop_autor_test = (
    df_train
    .groupby("autor")["id_lector"]
    .nunique()
)

caract_libros["n_lectores_distintos_autor"] = (
    caract_libros["autor"]
    .map(pop_autor_test)
    .fillna(0)
)

# Rating promedio del libro para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_libro_test = (
    df_train
    .groupby("id_libro")["rating"]
    .mean()
)

caract_libros["rating_prom_id_libro"] = (
    caract_libros["id_libro"]
    .map(means_libro_test)
    .fillna(media_global)
)

# Rating promedio del autor para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_autor_test = (
    df_train
    .groupby("autor")["rating"]
    .mean()
)

caract_libros["rating_prom_autor"] = (
    caract_libros["autor"]
    .map(means_autor_test)
    .fillna(media_global)
)

# Rating promedio del género para TEST ----> Se calcula sobre todo df_train, sin LOO.
means_genero_test = (
    df_train
    .groupby("genero_libro_agrupado")["rating"]
    .mean()
)

caract_libros["rating_prom_genero"] = (
    caract_libros["genero_libro_agrupado"]
    .map(means_genero_test)
    .fillna(media_global)
)

#iv. Comprobación: no puede quedar ningún nulo en lo que va al modelo.
nulos = caract_libros.isna().sum()
print("Nulos en caract_libros:")
print(nulos[nulos > 0] if nulos.sum() else "Ninguno.")
print("Libros:", len(caract_libros), "| Duplicados:", caract_libros["id_libro"].duplicated().sum())
print("Libros sin interacciones en train:", (caract_libros["frecuencia_libro"] == 0).sum())

Nulos en caract_libros:
autor                    78227
genero_libro_agrupado    78220
dtype: int64
Libros: 128743 | Duplicados: 0
Libros sin interacciones en train: 80676


In [17]:
#vi. Afinidad lector-autor para TEST ----> Se calcula sobre todo df_train, sin LOO.
afinidad_lector_autor_test = (
    df_train
    .groupby(["id_lector", "autor"])
    .agg(
        rating_prom_id_lector_autor=("rating", "mean"),
        n_interacciones_lector_autor=("rating", "count")
    )
    .reset_index()
)

#v. Afinidad lector-género para TEST ----> Se calcula sobre todo df_train, sin LOO.
afinidad_lector_genero_test = (
    df_train
    .groupby(["id_lector", "genero_libro_agrupado"])
    .agg(
        rating_prom_id_lector_genero_libro_agrupado=("rating", "mean"),
        n_interacciones_lector_genero=("rating", "count")
    )
    .reset_index()
)

In [18]:
#vii. Comprobación.
print(
    "Duplicados lectores:",
    caract_lector["id_lector"].duplicated().sum()
)

print(
    "Duplicados libros:",
    caract_libros["id_libro"].duplicated().sum()
)

print(
    "Duplicados lector-autor:",
    afinidad_lector_autor_test
    .duplicated(["id_lector", "autor"])
    .sum()
)

print(
    "Duplicados lector-género:",
    afinidad_lector_genero_test
    .duplicated(["id_lector", "genero_libro_agrupado"])
    .sum()
)

Duplicados lectores: 0
Duplicados libros: 0
Duplicados lector-autor: 0
Duplicados lector-género: 0


In [19]:
#c. Calculamos ranking final para producción.
#i. Lista donde almacenaremos las recomendaciones.
recomendaciones = []
total_lectores = len(id_lectores_predecir)

print("Comienza la predicción general.")
#ii. Recorro cada id_lector.
for i, id_lector in enumerate(id_lectores_predecir, start=1):

    print("\nLector: {} {}/{}".format(id_lector,i,total_lectores))
    #1. Retrieval.
    print("1. Retrieval.")
    libros_candidatos_a_recomendar = retrieval(id_lector)

    # Si no tiene candidatos, sigo con el próximo lector.
    if len(libros_candidatos_a_recomendar) == 0:
        continue

    #2. Feature engineering sobre todos los candidatos.
    print("2. Armo las features de los libros candidatos.")
    df_features_candidatos = feature_engineering_test(id_lector,libros_candidatos_a_recomendar)

    #3. Predigo el rating de cada candidato.
    print("3. Predigo sobre los libros candidatos su rating.")
    X = df_features_candidatos[features]
    predicciones = rf.predict(X)

    #4. Agrego la predicción al dataframe.
    df_features_candidatos["rating_predicho"] = predicciones

    #5. Ordeno de mayor a menor y me quedo con los 20 mejores.
    top_20 = (df_features_candidatos.sort_values("rating_predicho", ascending=False).head(20))

    #6. Guardo las recomendaciones.
    recomendaciones.append(top_20[["id_lector", "id_libro"]])
    print("4. Top 20 generado.")


#iii. Uno todas las recomendaciones.
df_recomendaciones = pd.concat(recomendaciones,ignore_index=True)

print("\nPredicción general finalizada.")
print("Cantidad de recomendaciones:",len(df_recomendaciones))

Comienza la predicción general.

Lector: 05-03-1970 1/832
1. Retrieval.
2. Armo las features de los libros candidatos.
3. Predigo sobre los libros candidatos su rating.
4. Top 20 generado.

Lector: 05131344k 2/832
1. Retrieval.
2. Armo las features de los libros candidatos.
3. Predigo sobre los libros candidatos su rating.
4. Top 20 generado.

Lector: 10155384625823370 3/832
1. Retrieval.
2. Armo las features de los libros candidatos.
3. Predigo sobre los libros candidatos su rating.
4. Top 20 generado.

Lector: 10155485526803149 4/832
1. Retrieval.
2. Armo las features de los libros candidatos.
3. Predigo sobre los libros candidatos su rating.
4. Top 20 generado.

Lector: 10155504435866506 5/832
1. Retrieval.
2. Armo las features de los libros candidatos.
3. Predigo sobre los libros candidatos su rating.
4. Top 20 generado.

Lector: 10155829844135513 6/832
1. Retrieval.
2. Armo las features de los libros candidatos.
3. Predigo sobre los libros candidatos su rating.
4. Top 20 generado.

#### 7. Exportación de las recomendaciones.

In [21]:
#a. Defino el número de versión (a mano).
version = "14"

In [22]:
#b. Exporto la versión de esta corrida.
df_recomendaciones.to_csv(f"./outputs/entregable_{version}.csv",index=False)

In [23]:
#c. Compruebo.
print("Lectores pedidos:", df_a_predecir["id_lector"].nunique())
print("Lectores entregados:", df_recomendaciones["id_lector"].nunique())
print(df_recomendaciones.groupby("id_lector").size().value_counts())

Lectores pedidos: 832
Lectores entregados: 832
20    832
Name: count, dtype: int64
